In [7]:
!pip install tensorflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 524.2/524.2 MB 6.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.5/126.5 kB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 12.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 72.9 MB/s eta 0:00:00ta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.9/22.9 MB 83.1 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 47.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 15.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 115.9 MB/s eta 0:00:0000:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 440.8/440.8 kB 51.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.9/78.9 kB 15.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 105.3 MB/s e

In [1]:
import zarr
from matplotlib import pyplot as plt
import numpy as np
import dask.array as da
import tensorflow as tf
#from torch import bfloat16

bfloat16 = bfloat16.as_numpy_dtype

In [2]:

aia_path = "/mnt/sdomlv2_full/sdomlv2.zarr"
aia_data = zarr.group(zarr.DirectoryStore(aia_path))

HALF_PRECISION = True
USE_BFLOAT = True

In [3]:
def copy_zarr_group(source_group, target_group):
    try:
        # Copy arrays (datasets) from the source group to the target group
        for array_name, zarr_array in source_group.items():
            shape = zarr_array.shape
            dtype = zarr_array.dtype
            target_group.create_dataset(array_name, shape=shape, dtype=dtype)
    except:

        # Recursively copy nested groups
        for group_name, nested_group in source_group.groups():
            new_nested_group = target_group.create_group(group_name)
            copy_zarr_group(nested_group, new_nested_group)

In [4]:
output_path = '/home/willfaw/data' if not HALF_PRECISION else '/home/willfaw/data16'
if USE_BFLOAT: 
    output_path += 'bfloat'

new_group = zarr.group(store=output_path)

copy_zarr_group(aia_data, new_group)

new_group.info

Name,/
Type,zarr.hierarchy.Group
Read-only,False
Store type,zarr.storage.DirectoryStore
No. members,11
No. arrays,0
No. groups,11
Groups,"2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020"


In [5]:
years = ['2010', '2011', '2012', '2013', '2014'] #, '2015', '2016', '2017', '2018', '2019', '2020']
wavelengths = ['131A', '1600A', '1700A', '171A', '193A', '211A', '304A', '335A', '94A']

#years = ['2010']
wavelengths = ['131A']
#wavelengths = ['1600A', '1700A', '171A', '193A', '211A', '304A', '335A', '94A']


In [6]:
# Hack: randomly sub-sample the aia data to speed up normalization calculation

def print_array_info(array):
        print(f"\tmean: {array.mean()}, max: {array.max()}, min: {array.min()}")
        print(f"\tarray size: {array.size}, memory size of one element: {array.itemsize}, total memory: {array.size * array.itemsize}")



print("activate hack")
fraction_size = 0.01
#years = ['2010']

for year in years: 
    for wavelength in wavelengths:
        print(year, wavelength)
        total_elements = aia_data[year][wavelength].shape[0]
        num_elements_to_select = int(total_elements * fraction_size)
        random_indices = np.random.choice(total_elements, num_elements_to_select, replace=False)
        print(f"Before: {aia_data[year][wavelength].shape[0]}, selcected {num_elements_to_select} elements. Writing ...")
        #new_zarr_array = aia_data[year][wavelength].get_coordinate_selection(random_indices)
        #new_zarr_array = da.from_array(aia_data[year][wavelength])[random_indices]
        new_zarr_array = aia_data[year][wavelength][random_indices]

        print_array_info(new_zarr_array)
        if HALF_PRECISION:
            new_zarr_array /= 2 
            if USE_BFLOAT:
                new_zarr_array = new_zarr_array.astype(bfloat16)
            else:
                new_zarr_array = new_zarr_array.astype(np.float16)
            print_array_info(new_zarr_array)
        new_group[year][wavelength] = new_zarr_array
        print(f"After: {new_group[year][wavelength].shape[0]}")
        print('-'*20)

activate hack
2010 131A
Before: 47116, selcected 471 elements. Writing ...
	mean: 3.287559986114502, max: 1693.1552734375, min: 0.0
	array size: 123469824, memory size of one element: 4, total memory: 493879296


TypeError: Cannot interpret 'torch.bfloat16' as a data type